# 04 第 2 節｜YOLO11 物件偵測與人工標註

**高中生教學版：3 組必做練習＋Optional。** 必做仍是讀取圖片、載入模型、單張推論；人工標註與微調直接執行現成範例，不要求自行撰寫。

**上課前：**將教材 ZIP 解壓縮到 Colab `/content/`，或將圖片放在 Notebook 可讀取的相同相對路徑。既有的 8 張授權圖片位於 `04_YOLO練習資料/images/`。

**標註工具（Optional）：**在電腦瀏覽器開啟 `04_標註工具.html`，可新增、重新命名或刪除任何類別（包括預設類別），也能選擇每張圖片屬於 Train／Val，再用滑鼠畫框。匯出 **`04_YOLO標註資料集.zip`** 後，Optional 會將圖片、標註及 `data.yaml` 一起匯入**獨立的** `04_YOLO學生標註資料/`。

**擴充照片：**教師可開啟 `04_教師取得40張練習圖片.ipynb`，在有網路的 Colab 取得額外 30 張 Train＋10 張 Val 的 Open Images V7 圖片。**本 ZIP 只內附原本 8 張**；下載完成後的圖片 ZIP 由教師另行取得。

**注意：**沒有指定目標的圖片可以 0 框；不必建立「背景」類別。照片即使擴充至數十張，也不等於足以評估真正的泛化能力。


## 0. 安裝套件（直接執行）


In [ ]:
# Install the Ultralytics YOLO package.
# 安裝 YOLO 物件偵測套件。
%pip -q install 'ultralytics>=8.3,<9'


## 0. 匯入套件（直接執行）


In [ ]:
# Import YOLO for object detection.
# 匯入 YOLO 模型。
from ultralytics import YOLO

# Import OpenCV to convert the display colors.
# 匯入 OpenCV 轉換色彩。
import cv2

# Import PyTorch to check GPU support.
# 匯入 PyTorch 檢查 GPU。
import torch

# Import image display tools.
# 匯入圖片顯示工具。
from PIL import Image
from IPython.display import display

# Import the file and ZIP helpers used in Optional.
# 匯入選做部分需要的檔案與壓縮工具。
from pathlib import Path
import zipfile

# Import helpers for dataset updates and class configuration.
# 匯入重建資料夾與讀取類別設定的工具。
import shutil
import yaml


## 練習 1｜查看一張圖片

先確認教材中的圖片存在。要看其他圖片，只需更改 `IMAGE_PATH`。

```python
# Choose one image to detect.
# 選擇要辨識的圖片。
IMAGE_PATH = '04_YOLO練習資料/images/train/train_02_cat.jpg'

# Display the original picture.
# 顯示原始圖片。
display(Image.open(IMAGE_PATH))
```

In [ ]:
# 請在此處根據範例練習程式碼


## 練習 2｜載入 YOLO11 預訓練模型

YOLO11 Nano 模型會在第一次執行時下載權重；這一步需要網路。

```python
# Load the pretrained YOLO11 Nano model.
# 載入 YOLO11 Nano 預訓練模型。
model = YOLO('yolo11n.pt')

# Check GPU availability.
# 檢查目前是否有 GPU。
print('GPU：', torch.cuda.is_available())
```


In [ ]:
# 請在此處根據範例練習程式碼


## 練習 3｜偵測物件並顯示方框

模型會回傳偵測方框、類別名稱及信心分數。

```python
# Detect objects in the image.
# 讓模型偵測圖片中的物件。
result = model.predict(IMAGE_PATH, conf=0.25, verbose=False)[0]

# Display the picture with bounding boxes.
# 顯示物件偵測框與類別。
display(Image.fromarray(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)))
```


In [ ]:
# 請在此處根據範例練習程式碼


## Optional A｜用滑鼠畫框，自訂類別與 Train／Val

這個 Optional **不用寫 Python 畫框程式**，直接使用教材內的 `04_標註工具.html`。

1. 在本機 Chrome、Edge 或 Safari 開啟 HTML，按「加入圖片」選取教材的 8 張圖片，或先由教師執行 `04_教師取得40張練習圖片.ipynb`，取得另外的 30 張訓練圖與 10 張驗證圖。
2. 選擇要標註的物件類別。原本有 `person`、`cat`、`cup`、`horse`、`rocket`、`clock`，**但全部都可以新增、修改名稱、刪除**；有畫框的類別刪除時會先詢問。
3. **每張圖片右側可自行指定 Train／Val**。例如驗證集應盡量包含訓練集也有的目標類別，而不是只用背景圖充當驗證資料。
4. 在圖片上拖曳滑鼠畫框，每一個需要偵測的物件各畫一框，完成後按「完成本張標註」。**沒有指定目標時可以 0 框，不用建立背景類別。**
5. 匯出 `04_YOLO標註資料集.zip`，其中會有重新分組後的 `images/train`、`images/val`、`labels/train`、`labels/val`、`data.yaml`、`classes.txt`。

**YOLO 格式：**每張圖片與 `.txt` 檔名相同，每一行為 `類別編號 x中心 y中心 寬 高`，座標範圍 0～1。類別編號從 0 開始，依匯出時的 `data.yaml` 決定。

**照片下載：**教師取得新圖片的 Notebook 是另一份直接執行的輔助教材。圖片會來自 Open Images V7，並記錄來源 ID；對外重新發布前應個別查證照片授權。

**備註：**原本提供的資料夾只是練習素材來源；人工標註後的 ZIP 包含重新分組的圖片，後續會解壓縮至獨立資料夾，避免與原來預設分組混在一起。資料集太小時，Train／Val 指標僅作流程示範。

官方格式：[Ultralytics YOLO Object Detection](https://docs.ultralytics.com/datasets/detect)。


### Optional｜確認匯出 ZIP 會包含的訓練目錄


In [ ]:
# Show the folder structure of a student-created dataset.
# 顯示學生標註資料集解壓縮後的資料夾。
print('訓練圖片：04_YOLO學生標註資料/images/train/')
print('訓練標註：04_YOLO學生標註資料/labels/train/')
print('驗證圖片：04_YOLO學生標註資料/images/val/')
print('驗證標註：04_YOLO學生標註資料/labels/val/')
print('類別設定：04_YOLO學生標註資料/data.yaml')


## Optional B｜將完整的標註資料集上傳 Colab

1. 在 HTML 標註工具匯出 **`04_YOLO標註資料集.zip`**（已包含圖片、標註與 `data.yaml`）。
2. 在 Colab 執行上傳與解壓縮 Cell；解壓縮後會得到新的 **`04_YOLO學生標註資料/`**，不會混入原始範例的 Train／Val。
3. **重新解壓縮時會清除同名的 `04_YOLO學生標註資料/` 舊內容**，所以請另外保留下載的 ZIP。若在本機 VS Code 執行，可手動將 ZIP 解壓到此資料夾。


### Optional｜Colab 上傳標註 ZIP


In [ ]:
# Upload the YOLO dataset ZIP from your computer.
# 從自己的電腦上傳完整 YOLO 資料集 ZIP。
from google.colab import files
uploaded = files.upload()


### Optional｜解壓縮圖片、標註與 data.yaml（直接執行）


In [ ]:
# Choose the uploaded dataset ZIP.
# 指定剛才匯出的完整標註資料集壓縮檔。
LABEL_ZIP = '04_YOLO標註資料集.zip'

# Set a separate folder for the new dataset.
# 使用獨立資料夾存放新資料，避免混入原本的範例圖片。
DATA_DIR = Path('04_YOLO學生標註資料')

# Clear the previous practice dataset if it exists.
# 若此資料夾已經存在，先清除舊的練習資料。
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

# Extract the complete dataset.
# 解壓縮圖片、標註和類別設定檔。
with zipfile.ZipFile(LABEL_ZIP) as z:
    z.extractall(DATA_DIR)

# Show the imported classes and image counts.
# 顯示自訂類別與訓練、驗證圖片的數量。
print((DATA_DIR / 'data.yaml').read_text(encoding='utf-8'))
print('Train：', len(list((DATA_DIR / 'images/train').iterdir())), '張')
print('Val：', len(list((DATA_DIR / 'images/val').iterdir())), '張')


### Optional｜查看一張圖片的 YOLO 標註


In [ ]:
# Choose an annotated image from either split.
# 從訓練或驗證資料中選擇一張圖片。
image_path = next(p for p in (DATA_DIR / 'images').rglob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp'])

# Find the label file with the same name and split.
# 根據圖片所在群組及檔名，找到對應的標註檔。
label_path = DATA_DIR / 'labels' / image_path.parent.name / (image_path.stem + '.txt')

# Show the image and its label text.
# 顯示圖片和標註內容。
display(Image.open(image_path))
print('標註檔：', label_path)
print(label_path.read_text(encoding='utf-8') or '（0 個框：背景圖片）')


## Optional C｜微調 YOLO（有時間與 GPU 時再執行）

請先完成標註，並解壓縮至 `04_YOLO學生標註資料/`。下方會根據**匯出的 `data.yaml`** 確認類別編號，因此自訂類別也能使用。

提醒：原本只有 8 張圖片；教師可另下載 40 張不同原圖，但每個類別仍只有少量照片，**不能把練習指標當成可靠的模型成效**；標註若有遺漏，還可能把物件誤當背景。只將微調視為操作流程練習。


### Optional｜檢查圖片、標註與動態類別設定（直接執行）


In [ ]:
# Read the student dataset and its class names.
# 讀取學生標註的資料集與類別設定。
DATA_DIR = Path('04_YOLO學生標註資料')

# Get the class names from data.yaml.
# 從 data.yaml 取得目前的類別編號。
class_names = yaml.safe_load((DATA_DIR / 'data.yaml').read_text(encoding='utf-8'))['names']

# Check all images and annotations.
# 檢查兩組圖片與標註內容是否配對。
for split in ['train', 'val']:
    for image_path in (DATA_DIR / 'images' / split).iterdir():
        if image_path.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.webp']:
            continue
        label_path = DATA_DIR / 'labels' / split / (image_path.stem + '.txt')
        if not label_path.exists():
            raise FileNotFoundError(f'找不到標註檔：{label_path}')
        for line in label_path.read_text(encoding='utf-8').splitlines():
            parts = line.split()
            if len(parts) != 5:
                raise ValueError(f'標註格式應為 5 個數字：{label_path}')
            values = [float(p) for p in parts]
            if values[0] != int(values[0]) or int(values[0]) not in class_names:
                raise ValueError(f'類別編號錯誤：{label_path}')
            if any(not 0 <= v <= 1 for v in values[1:]) or values[3] <= 0 or values[4] <= 0:
                raise ValueError(f'框的座標錯誤：{label_path}')

# Show the verification result.
# 顯示檢查完成訊息。
print('標註格式正確。類別：', class_names)


### Optional｜使用學生自訂的資料進行 YOLO 微調


In [ ]:
# Configure the number of training epochs.
# 指定訓練輪數。
EPOCHS = 3

# Choose GPU when available.
# 有 GPU 時使用 GPU，否則使用 CPU。
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# Load the pretrained model to fine-tune.
# 載入預訓練模型進行微調。
model = YOLO('yolo11n.pt')

# Train using our own images and labels.
# 使用自己標註的影像與文字檔進行訓練。
model.train(data=str((DATA_DIR / 'data.yaml').resolve()), epochs=EPOCHS, imgsz=416, batch=2, device=DEVICE, project='04_訓練成果', name='example', exist_ok=True)


### Optional｜使用自己微調後的模型預測一張圖片


In [ ]:
import glob

# Find validation images.
# 找出驗證集的圖片。
val_images = sorted(glob.glob('04_YOLO學生標註資料/images/val/*'))

# Select the first validation image.
# 選擇第一張驗證圖片。
IMAGE_PATH = val_images[0]

# Load the checkpoint saved after fine-tuning.
# 載入微調完成的模型權重。
fine = YOLO('04_訓練成果/example/weights/best.pt')

# Predict one image.
# 使用微調後的模型偵測圖片。
answer = fine.predict(IMAGE_PATH, verbose=False)[0]

# Display the result.
# 顯示偵測結果。
display(Image.fromarray(cv2.cvtColor(answer.plot(), cv2.COLOR_BGR2RGB)))

**課堂回顧：**物件偵測會輸出類別與方框；標註時每張圖片需要對應同名 `.txt`；`data.yaml` 決定類別名稱和 Train／Val 位置。新增類別不會讓預訓練模型直接認得該類別，必須蒐集、標註足夠的訓練圖片再訓練。

內附 8 張圖片與可另行下載的 40 張練習圖片，都以操作流程與人工標註為主。背景照片可以有 0 個框，但不能只用背景圖片代表模型的驗證能力。Colab 的 `/content/` 是暫存空間，請把 HTML 工具匯出的 ZIP 保存到自己的電腦。
